# Chapter 11 – Training Deep Neural Networks

## 1. Vanishing and Exploding Gradients & Initialization

Jaringan saraf yang dalam (*deep networks*) sering kali sulit dilatih karena gradien dapat:
- **mengecil secara drastis (vanishing gradients)**, atau
- **membesar tidak terkendali (exploding gradients)**

ketika gradien tersebut dipropagasikan mundur melalui banyak lapisan selama proses **backpropagation**.

Masalah ini menjadi sangat parah ketika menggunakan fungsi aktivasi klasik seperti **sigmoid** atau **tanh**, terutama jika dikombinasikan dengan **inisialisasi bobot yang tidak tepat**. Akibatnya, lapisan-lapisan awal (*lower layers*) hampir tidak menerima sinyal pembelajaran dan gagal belajar secara efektif.

---

### Weight Initialization

Untuk mengatasi masalah tersebut, **Glorot dan Bengio** mengusulkan skema **Xavier (Glorot) initialization**, di mana bobot diinisialisasi secara acak dengan **varians tertentu** sehingga:
- varians output setiap layer kira-kira sama dengan varians input,
- sinyal aktivasi dan gradien tetap stabil di seluruh jaringan.

Pendekatan ini membantu mencegah vanishing maupun exploding gradients pada tahap awal pelatihan.

---

### Initialization Schemes by Activation Function

Berbagai fungsi aktivasi memerlukan skema inisialisasi yang berbeda agar bekerja secara optimal, antara lain:
- **Glorot (Xavier) initialization** untuk *tanh*, *sigmoid*, dan *softmax*
- **He initialization** untuk **ReLU** dan variannya
- **LeCun initialization** untuk **SELU**

Parameter dan perbandingan skema inisialisasi ini biasanya dirangkum dalam referensi seperti **Tabel 11-1** pada literatur pembelajaran mendalam.


## **Example: He, Glorot, LeCun Initialization in Keras**

In [1]:
from tensorflow import keras

# He initialization (untuk ReLU dan variannya)
he_layer = keras.layers.Dense(
    10, activation="relu", kernel_initializer="he_normal"
)

# Glorot (default Keras) dengan VarianceScaling
he_avg_init = keras.initializers.VarianceScaling(
    scale=2., mode="fan_avg", distribution="uniform"
)
glorot_like_layer = keras.layers.Dense(
    10, activation="sigmoid", kernel_initializer=he_avg_init
)

# LeCun initialization (untuk SELU)
lecun_layer = keras.layers.Dense(
    10, activation="selu", kernel_initializer="lecun_normal"
)


## 2. Advanced Activation Functions (ReLU, ELU, SELU)

Fungsi aktivasi klasik seperti **sigmoid** cenderung mengalami **saturasi** (gradien mendekati nol) untuk nilai input yang besar. Hal ini memperparah masalah **vanishing gradients** pada jaringan saraf yang dalam. Penelitian selanjutnya menunjukkan bahwa fungsi aktivasi **non-saturating**, seperti **ReLU**, jauh lebih stabil dan efisien ketika digunakan pada *deep networks*.

---

### Variants of ReLU

Beberapa variasi ReLU dikembangkan untuk mengatasi kelemahan ReLU standar, khususnya masalah *dead neurons*:

- **Leaky ReLU**  
  Mempertahankan gradien kecil namun **non-nol** pada sisi negatif, sehingga neuron tidak berhenti belajar.

- **Randomized ReLU (RReLU)**  
  Menggunakan kemiringan acak pada sisi negatif selama training, yang bertindak sebagai bentuk regularisasi.

- **Parametric ReLU (PReLU)**  
  Memperlakukan kemiringan sisi negatif sebagai parameter yang dapat dipelajari oleh model.


## **Example: Using Leaky ReLU, PReLU, SELU**

In [2]:
from tensorflow import keras

# Leaky ReLU sesudah Dense
model = keras.models.Sequential([
    keras.layers.Dense(10, kernel_initializer="he_normal"),
    keras.layers.LeakyReLU(alpha=0.2),
])

# PReLU (alpha dipelajari)
model_prelu = keras.models.Sequential([
    keras.layers.Dense(10, kernel_initializer="he_normal"),
    keras.layers.PReLU(),
])

# SELU + LeCun normal untuk self-normalizing dense net
selu_layer = keras.layers.Dense(
    10, activation="selu", kernel_initializer="lecun_normal"
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


## 3. Batch Normalization & Gradient Clipping

**Batch Normalization (BN)** menambahkan operasi normalisasi pada input suatu layer dengan cara:
- men-*zero-center* dan men-*scale* aktivasi berdasarkan statistik mini-batch,
- kemudian melakukan *scale* dan *shift* ulang menggunakan parameter yang dapat dilatih, yaitu **γ (gamma)** dan **β (beta)**.

Teknik ini membantu **menstabilkan distribusi aktivasi antar layer**, sehingga:
- mempercepat konvergensi pelatihan,
- mengurangi sensitivitas terhadap inisialisasi bobot,
- serta bertindak sebagai bentuk **regularisasi** implisit.

---

### Placement of Batch Normalization

Batch Normalization umumnya ditempatkan:
- **sebelum aktivasi** pada *hidden layer*, atau
- **sesudah aktivasi**, tergantung pada arsitektur dan praktik implementasi.

Jika BN digunakan pada **layer pertama**, teknik ini bahkan dapat menggantikan proses *standardization* pada input data.

---

### Gradient Clipping

Untuk mengatasi masalah **exploding gradients**, teknik lain yang sering digunakan adalah **gradient clipping**.  
Metode ini membatasi nilai gradien agar tidak melebihi **ambang batas (threshold)** tertentu, baik dengan:
- **value clipping**, yaitu memotong setiap elemen gradien secara individual, atau
- **norm clipping**, yaitu membatasi norma gradien secara keseluruhan.

Gradient clipping sangat umum digunakan pada pelatihan **recurrent neural networks** dan model dalam lainnya yang rentan terhadap gradien besar.


## **Example: Batch Normalization & Gradient Clipping in Keras**

In [3]:
from tensorflow import keras

# Model dengan Batch Normalization setelah setiap hidden layer
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(300, activation="elu", kernel_initializer="he_normal"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(10, activation="softmax")
])

# Optimizer dengan gradient clipping by value
optimizer = keras.optimizers.SGD(learning_rate=0.01, clipvalue=1.0)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## 4. Transfer Learning & Unsupervised Pretraining

Untuk **deep neural network** berukuran besar, melatih model dari awal (*training from scratch*) sering kali tidak efisien, terutama ketika **data berlabel terbatas**. **Transfer learning** menawarkan solusi dengan memanfaatkan model yang telah dilatih sebelumnya pada tugas atau domain yang serupa.

Pada pendekatan ini, **layer bawah** jaringan (yang berfungsi sebagai *feature extractor*) digunakan kembali, sementara **output layer** diganti dan dilatih ulang agar sesuai dengan tugas baru.

---

### Practical Workflow

Dalam praktik, proses transfer learning biasanya dilakukan sebagai berikut:
1. **Reuse layer bawah** dari model pra-latih.
2. **Bekukan (freeze)** layer yang digunakan kembali agar bobot awal tidak langsung rusak oleh gradien besar dari layer baru.
3. Latih **output layer baru** selama beberapa epoch.
4. **Unfreeze** sebagian *upper layers* dan lakukan **fine-tuning** dengan *learning rate* yang lebih kecil.

Strategi ini menjaga stabilitas pembelajaran sekaligus memungkinkan adaptasi bertahap terhadap tugas baru.

---

### Unsupervised and Self-Supervised Pretraining

Jika tidak tersedia model pra-latih yang sesuai, dapat digunakan **unsupervised pretraining** atau **self-supervised learning** untuk mengekstraksi representasi fitur yang berguna, misalnya melalui:
- **Autoencoders**
- **Generative Adversarial Networks (GANs)**
- Tugas bantu (*auxiliary tasks*) berbasis self-supervision

Layer bawah hasil pretraining ini kemudian digunakan kembali untuk tugas utama yang berlabel.


## **Example: Transfer Learning with Keras**

In [ ]:
from tensorflow import keras

# Load model A (sudah dilatih)
model_A = keras.models.load_model("my_model_A.h5")

# Reuse semua layer kecuali output, lalu tambah output baru biner
model_B_on_A = keras.models.Sequential(model_A.layers[:-1])
model_B_on_A.add(keras.layers.Dense(1, activation="sigmoid"))

# Bekukan layer lama
for layer in model_B_on_A.layers[:-1]:
    layer.trainable = False

model_B_on_A.compile(loss="binary_crossentropy",
                     optimizer="sgd",
                     metrics=["accuracy"])

# Warm-up training
history = model_B_on_A.fit(X_train_B, y_train_B, epochs=4,
                           validation_data=(X_valid_B, y_valid_B))

# Unfreeze dan fine-tune dengan learning rate kecil
for layer in model_B_on_A.layers[:-1]:
    layer.trainable = True

optimizer = keras.optimizers.SGD(learning_rate=1e-4)
model_B_on_A.compile(loss="binary_crossentropy",
                     optimizer=optimizer,
                     metrics=["accuracy"])

history_fine = model_B_on_A.fit(X_train_B, y_train_B, epochs=16,
                                validation_data=(X_valid_B, y_valid_B))


## 5. Faster Optimizers & Learning Rate Scheduling

Selain arsitektur jaringan, **kecepatan dan kualitas pelatihan** sangat dipengaruhi oleh pemilihan **optimizer** dan **learning rate schedule**. Strategi optimisasi yang tepat dapat mempercepat konvergensi sekaligus meningkatkan performa generalisasi model.

---

### Optimizers

Beberapa optimizer yang umum dan penting dalam pelatihan *deep neural networks* antara lain:

- **Momentum & Nesterov Accelerated Gradient (NAG)**  
  Menambahkan vektor momentum pada *Gradient Descent* sehingga langkah optimisasi dapat “menggelinding” lebih cepat dan menavigasi lembah sempit dengan lebih efisien.  
  **NAG** menghitung gradien pada posisi parameter yang telah digeser oleh momentum, sehingga sering menghasilkan konvergensi yang lebih cepat.

- **AdaGrad**  
  Menggunakan *learning rate* adaptif yang mengecil lebih cepat pada dimensi dengan gradien besar. Pendekatan ini berguna untuk fitur jarang (*sparse features*), tetapi pada *deep networks* sering kali menyebabkan *learning rate* menjadi terlalu kecil dan pelatihan berhenti terlalu dini.

- **RMSProp**  
  Merupakan pengembangan dari AdaGrad yang hanya mempertimbangkan gradien terbaru menggunakan *exponential decay*. Optimizer ini umumnya lebih stabil dan lebih cocok untuk pelatihan jaringan saraf dalam.

- **Adam & Nadam**  
  Menggabungkan konsep **momentum** dan **RMSProp** dengan estimasi rata-rata (*first moment*) dan varians (*second moment*) gradien secara eksponensial.  
  **Adam** dengan parameter default (`lr = 0.001`, `β1 = 0.9`, `β2 = 0.999`) sering memberikan hasil yang baik tanpa tuning yang intensif.  
  **Nadam** merupakan varian Adam yang menambahkan teknik **Nesterov momentum** untuk konvergensi yang lebih cepat.

---

### Learning Rate Scheduling

Penggunaan **learning rate schedule** sering kali menghasilkan konvergensi yang lebih cepat dan generalisasi yang lebih baik dibandingkan *learning rate* konstan. Beberapa pendekatan yang umum digunakan meliputi:
- **Exponential decay**
- **Performance-based scheduling** (misalnya *ReduceLROnPlateau*)
- **1cycle policy**

Strategi ini memungkinkan model untuk belajar cepat pada tahap awal pelatihan dan melakukan penyesuaian yang lebih halus pada tahap akhir.


## **Example: Optimizers and LR Scheduling in Keras**

In [ ]:
from tensorflow import keras

# Adam optimizer
adam_opt = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)

# RMSProp optimizer
rms_opt = keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9)

# SGD with Momentum + Nesterov
nag_opt = keras.optimizers.SGD(learning_rate=0.001, momentum=0.9, nesterov=True)

model.compile(loss="sparse_categorical_crossentropy",
              optimizer=adam_opt,
              metrics=["accuracy"])

# Exponential learning rate decay (callback)
def exponential_decay_fn(epoch):
    return 0.01 * 0.1**(epoch / 20)

lr_scheduler = keras.callbacks.LearningRateScheduler(exponential_decay_fn)

history = model.fit(X_train_scaled, y_train,
                    epochs=50,
                    validation_data=(X_valid_scaled, y_valid),
                    callbacks=[lr_scheduler])


## 6. Regularization: L1/L2, Dropout, MC Dropout, Max-Norm

Deep neural networks dengan jutaan parameter sangat rentan terhadap **overfitting**, sehingga teknik **regularisasi** menjadi komponen penting dalam proses pelatihan. Selain *early stopping* dan efek regularisasi implisit dari **Batch Normalization**, beberapa metode regularisasi utama dibahas pada bagian ini.

---

### L1 and L2 Regularization

**L1 dan L2 regularization** menambahkan penalti terhadap norma bobot ke dalam fungsi *loss*:
- **L2 regularization** (weight decay) mengecilkan bobot secara halus dan kontinu, sehingga membantu mengontrol kompleksitas model.
- **L1 regularization** mendorong **sparsity**, yaitu banyak bobot menjadi nol, yang dapat meningkatkan interpretabilitas dan efisiensi model.

---

### Dropout

**Dropout** merupakan teknik regularisasi di mana, selama proses training, setiap neuron (kecuali neuron output) memiliki probabilitas **p** untuk “dijatuhkan” (*output = 0*).

Pendekatan ini:
- memaksa jaringan untuk menyebarkan representasi informasi,
- mencegah ketergantungan berlebihan pada neuron tertentu,
- serta bertindak seperti **ensemble** dari banyak subnet yang berbeda.

---

### Monte Carlo (MC) Dropout

**MC Dropout** memperluas konsep dropout ke tahap inferensi. Dropout tetap diaktifkan dan model dijalankan beberapa kali untuk input yang sama, kemudian hasil prediksi dirata-ratakan.

Teknik ini:
- sering meningkatkan akurasi prediksi,
- memungkinkan estimasi **ketidakpastian prediksi** (*predictive uncertainty*),
- berguna pada aplikasi dengan risiko tinggi.

---

### Max-Norm Regularization

**Max-Norm regularization** membatasi norma vektor bobot agar tidak melebihi radius tertentu **r**.

Metode ini:
- membantu mengendalikan kompleksitas model,
- menstabilkan gradien selama pelatihan,
- sering dikombinasikan dengan dropout untuk hasil yang lebih robust.


## **Example: L2, Dropout, MC Dropout, Max-Norm**

In [ ]:
from tensorflow import keras
from functools import partial
import numpy as np

# L2-regularized dense layer helper
RegularizedDense = partial(
    keras.layers.Dense,
    activation="elu",
    kernel_initializer="he_normal",
    kernel_regularizer=keras.regularizers.l2(0.01)
)

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dropout(rate=0.2),
    RegularizedDense(300),
    keras.layers.Dropout(rate=0.2),
    RegularizedDense(100),
    keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(10, activation="softmax",
                       kernel_initializer="glorot_uniform",
                       kernel_constraint=keras.constraints.max_norm(1.0))
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])

history = model.fit(X_train_scaled, y_train,
                    epochs=30,
                    validation_data=(X_valid_scaled, y_valid))

# MC Dropout: rata-rata 100 prediksi dengan dropout aktif
y_probas = np.stack([
    model(X_test_scaled, training=True)  # penting: training=True
    for _ in range(100)
])
y_proba = y_probas.mean(axis=0)
y_pred = np.argmax(y_proba, axis=1)


## 7. Practical Default Configurations

Bab ini menutup pembahasan dengan beberapa **konfigurasi default praktis** yang dapat dijadikan titik awal dalam melatih *deep neural networks*.

---

### General Deep Neural Networks

Untuk jaringan saraf dalam secara umum, konfigurasi yang sering bekerja dengan baik adalah:
- **He initialization** dengan fungsi aktivasi **ELU**
- **Batch Normalization** jika jaringan cukup dalam
- **Early stopping**, dengan tambahan **L2 regularization** bila diperlukan
- Optimizer berbasis momentum, seperti **Momentum**, **RMSProp**, atau **Nadam**
- **Learning rate schedule 1cycle** untuk mempercepat konvergensi dan meningkatkan generalisasi

---

### Self-Normalizing Dense Networks

Untuk arsitektur *dense feedforward* yang dirancang agar bersifat *self-normalizing*, konfigurasi yang direkomendasikan meliputi:
- **LeCun initialization** dengan fungsi aktivasi **SELU**
- **Tanpa Batch Normalization**
- **Alpha Dropout** jika regularisasi tambahan diperlukan
- Optimizer dan *learning rate scheduling* yang serupa dengan konfigurasi umum

Pendekatan ini menjaga **mean** dan **standar deviasi** aktivasi tetap stabil di seluruh jaringan.

---

### Data-Efficient Training Strategies

Ketika **data berlabel terbatas**, sangat dianjurkan untuk memanfaatkan:
- **Transfer learning**
- **Unsupervised atau self-supervised pretraining**
- **Auxiliary tasks** untuk membantu pembelajaran representasi

Strategi-strategi ini secara signifikan meningkatkan efisiensi pelatihan dan performa model.
